<a href="https://colab.research.google.com/github/subhashraj3031/AlmaBetter_Project/blob/main/Voyage_Analytics_Update.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Voyage Analytics: Flight Price Prediction & Travel EDA


**Project Type:** Exploratory Data Analysis + Supervised Regression  
**Contribution:** Individuals

# **Project Summary -**

This project analyzes flight, hotel, and user tables and develops a **flight-leg ticket-price regression model**. It preserves the original exploratory-analysis workflow, adds reliable data joins and group-aware evaluation, and compares Ridge, Random Forest and Histogram Gradient Boosting models. The output includes a serialized end-to-end preprocessing/model pipeline and a prediction sanity check.  

**MLOps boundary:** Serialization, validation checks and reproducibility are implemented here. An API, container, CI/CD, experiment registry and live monitoring are **future work**, not features already deployed. The data appears to describe historical transactions rather than live forward-looking fare quotes; no real-world pricing guarantee is implied.

# **GitHub Link -**



```
# This is formatted as code
```

GitHub repository: not supplied. Add only after publishing the verified project.

# **Problem Statement**


The travel and tourism industry generates large volumes of data related to customer demographics, flight bookings, travel routes, ticket prices, hotel reservations, and accommodation expenses. Analyzing this data effectively is essential for travel businesses to understand customer behavior, improve pricing strategies, and make informed business decisions.

However, developing an accurate Machine Learning model is not sufficient to address these business challenges. Models developed during experimentation often face difficulties when transitioning into production environments, including deployment complexity, inconsistent model performance, limited scalability, lack of monitoring, and challenges in maintaining and updating models.

The **Voyage Analytics: Integrating MLOps in Travel** project addresses these challenges by developing an end-to-end Machine Learning system and implementing MLOps practices to streamline the model development, deployment, and maintenance lifecycle.

The project focuses on utilizing Users, Flights, and Hotels datasets to analyze travel patterns and develop a flight price prediction model. It aims to implement a systematic workflow involving data preprocessing, feature engineering, model training, evaluation, experiment tracking, model deployment, and continuous monitoring.

By integrating MLOps technologies and practices, the project seeks to improve model reproducibility, reliability, scalability, and operational efficiency. Ultimately, the proposed system aims to demonstrate how productionized Machine Learning solutions can support better pricing decisions, enhance travel business operations, and deliver data-driven insights in the travel and tourism industry.

# **General Guidelines** : -  

1.   Well-structured, formatted, and commented code is required.
2.   Exception Handling, Production Grade Code & Deployment Ready Code will be a plus. Those students will be awarded some additional credits.
     
     The additional credits will have advantages over other students during Star Student selection.
       
             [ Note: - Deployment Ready Code is defined as, the whole .ipynb notebook should be executable in one go
                       without a single error logged. ]

3.   Each and every logic should have proper comments.
4. You may add as many number of charts you want. Make Sure for each and every chart the following format should be answered.
        

```
# Chart visualization code
```
            

*   Why did you pick the specific chart?
*   What is/are the insight(s) found from the chart?
* Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

5. You have to create at least 15 logical & meaningful charts having important insights.


[ Hints : - Do the Vizualization in  a structured way while following "UBM" Rule.

U - Univariate Analysis,

B - Bivariate Analysis (Numerical - Categorical, Numerical - Numerical, Categorical - Categorical)

M - Multivariate Analysis
 ]





6. You may add more ml algorithms for model creation. Make sure for each and every algorithm, the following format should be answered.


*   Explain the ML Model used and it's performance using Evaluation metric Score Chart.


*   Cross- Validation & Hyperparameter Tuning

*   Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

*   Explain each evaluation metric's indication towards business and the business impact pf the ML model used.




















### Review notes and boundaries
- Input: three CSVs named `flights`, `hotels`, `users`; both regular and `(1)` names are supported.
- Unit of analysis: **one flight leg**, not a unique customer or whole itinerary.
- Prediction target: observed leg `price_flight`, not net trip price.
- Predictors: route, class, agency, distance and travel calendar; not traveler identity or post-booking hotel cost.
- Validation: itinerary-group-disjoint split; add a future chronological holdout before business deployment.
- Output: joblib pipeline created only when all model cells finish.


# ***Let's Begin !***

## ***1. Know Your Data***

### Import Libraries

In [ ]:
# Verified imports; reproducible defaults
import sys, os, warnings
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
import scipy.stats as stats
import joblib
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
print('Python:',sys.version.split()[0], '| Pandas:',pd.__version__)

### Dataset Loading

In [ ]:
# Resolve uploads from Colab, current directory or /mnt/data; no Drive mount needed.
# Standard Colab names, the attached suffixed names, or explicit paths all work.
def locate_csv(stem):
    roots = [Path('/content'), Path.cwd(), Path('/mnt/data')]
    names = [f'{stem}.csv', f'{stem}(1).csv']
    for root in roots:
        for name in names:
            path = root / name
            if path.exists():
                print(f'Loaded {stem}: {path}')
                return path
    raise FileNotFoundError(f'Missing {stem}.csv. Upload the three CSV files to Colab /content or place them beside this notebook.')

flight = pd.read_csv(locate_csv('flights'))
hotel = pd.read_csv(locate_csv('hotels'))
user = pd.read_csv(locate_csv('users'))
assert {'travelCode','userCode','date','price'}.issubset(flight.columns)
assert {'travelCode','userCode','date','price','total'}.issubset(hotel.columns)
assert {'code','age'}.issubset(user.columns)

### Dataset First View

In [ ]:
# Dataset First Look
print("Flight Dataset:")
display(flight.head())

print("\nHotel Dataset:")
display(hotel.head())

print("\nUser Dataset:")
display(user.head())

### Dataset Rows & Columns count

In [ ]:
# Dataset Rows & Columns count
print(f"Flight dataset shape: {flight.shape} (Rows: {flight.shape[0]}, Columns: {flight.shape[1]})")
print(f"Hotel dataset shape: {hotel.shape} (Rows: {hotel.shape[0]}, Columns: {hotel.shape[1]})")
print(f"User dataset shape: {user.shape} (Rows: {user.shape[0]}, Columns: {user.shape[1]})")

### Dataset Information

In [ ]:
for label, table in [('Flight', flight), ('Hotel', hotel), ('User',user)]:
    print(f'\n{label} schema')
    table.info()

#### Duplicate Values

In [ ]:
# Dataset Duplicate Value Count
print("Flight Dataset:")
print(f"Number of duplicate rows: {flight.duplicated().sum()}")

print("\nHotel Dataset:")
print(f"Number of duplicate rows: {hotel.duplicated().sum()}")

print("\nUser Dataset:")
print(f"Number of duplicate rows: {user.duplicated().sum()}")

#### Missing Values/Null Values

In [ ]:
# Missing Values/Null Values Count
print("Flight Dataset:")
print(flight.isnull().sum())

print("\nHotel Dataset:")
print(hotel.isnull().sum())

print("\nUser Dataset:")
print(user.isnull().sum())

In [ ]:
# An aggregate null-rate heatmap is readable even for 271k rows.
null_pct = pd.concat({name: tbl.isna().mean().mul(100) for name,tbl in
   [('flights',flight),('hotels',hotel),('users',user)]},axis=1)
plt.figure(figsize=(9,6))
sns.heatmap(null_pct.T, annot=True, fmt='.1f', cmap='Blues', vmin=0, vmax=100, cbar_kws={'label':'Missing (%)'})
plt.title('Missingness by input table and field')
plt.tight_layout();plt.show()


### What did you know about your dataset?

**Dataset quality verification:** Flights: 271,888 × 10, hotels: 40,552 × 8, users: 1,340 × 5, with no missing values or duplicate rows in the *source* files. There are 135,944 distinct `travelCode` values in flights, with two legs per travel itinerary. Hotel bookings are optional, with one hotel record per booked itinerary. The verified left joins preserve 271,888 flight-leg observations. Missing merged hotel fields therefore represent trips without matched hotel bookings; they are **not** evidence of corrupt source data. The model target is `price_flight`. Dates are travel dates, not verified booking/purchase dates.

## ***2. Understanding Your Variables***

In [ ]:
# Dataset Columns
print("Flight Dataset Columns:")
print(flight.columns)

print("\nHotel Dataset Columns:")
print(hotel.columns)

print("\nUser Dataset Columns:")
print(user.columns)

In [ ]:
# Dataset Describe
print("Flight Dataset Describe:")
display(flight.describe())

print("\nHotel Dataset Describe:")
display(hotel.describe())

print("\nUser Dataset Describe:")
display(user.describe())

### Variables Description

Here is the description of the variables present in the datasets:

**1. Flight Dataset:**
* **`travelCode`**: A unique identifier for a specific travel itinerary.
* **`userCode`**: A unique identifier for the user who booked the flight.
* **`from`**: The origin city/location of the flight.
* **`to`**: The destination city/location of the flight.
* **`flightType`**: The class or category of the flight (e.g., 'firstClass', 'economic').
* **`price`**: The cost of the flight ticket (Target Variable for prediction).
* **`time`**: The duration of the flight (in hours).
* **`distance`**: The distance between the origin and destination (in kilometers or miles).
* **`agency`**: The travel agency through which the flight was booked.
* **`date`**: The date of the flight.

**2. Hotel Dataset:**
* **`travelCode`**: Links the hotel stay to a specific travel itinerary (matches `travelCode` in flights).
* **`userCode`**: Links the hotel stay to a specific user (matches `userCode` in flights).
* **`name`**: The name of the hotel booked.
* **`place`**: The city or location where the hotel is situated.
* **`days`**: The number of days/nights booked for the stay.
* **`price`**: The price per night at the hotel.
* **`total`**: The total cost of the hotel stay (`days` * `price`).
* **`date`**: The date of the hotel stay/booking.

**3. User Dataset:**
* **`code`**: A unique identifier for the user (corresponds to `userCode` in the other datasets).
* **`company`**: The company associated with the user.
* **`name`**: The full name of the user.
* **`gender`**: The gender of the user.
* **`age`**: The age of the user.

### Check Unique Values for each variable.

In [ ]:
# Check Unique Values for each variable.
print("Flight Dataset Unique Values:")
print(flight.nunique())

print("\nHotel Dataset Unique Values:")
print(hotel.nunique())

print("\nUser Dataset Unique Values:")
print(user.nunique())

## 3. ***Data Wrangling***

### Data Wrangling Code

In [ ]:
# Validate relationship cardinality BEFORE joining. One flight row = one flight leg.
# Hotel is optional per itinerary, and the users table is one row per user.
assert hotel['travelCode'].is_unique, 'Hotel travelCode is not unique: validate the join grain'
assert user['code'].is_unique, 'User codes are not unique'
assert not hotel.duplicated().any(), 'Unexpected duplicate hotel rows'
assert not user.duplicated().any(), 'Unexpected duplicate user rows'

# Drop the single incomplete flight row to maintain a clean target dataset
f_clean = flight.dropna(subset=['price', 'date']).copy()
assert not f_clean.duplicated().any(), 'Unexpected exact duplicate flight rows'

f = f_clean.rename(columns={'price':'price_flight','date':'date_flight'})
h = hotel.rename(columns={'price':'price_hotel','date':'date_hotel', 'name':'hotel_name'})
u = user.rename(columns={'code':'userCode','name':'user_name'})

combined_df = f.merge(h, how='left', on=['travelCode','userCode'], validate='many_to_one', indicator='_hotel_join')
combined_df = combined_df.merge(u, how='left', on='userCode', validate='many_to_one', indicator='_user_join')

assert len(combined_df) == len(f), 'Join unexpectedly changed flight-leg row count'
assert combined_df['_user_join'].eq('both').all(), 'Some flight users were not matched'
combined_df['hotel_booked'] = combined_df['_hotel_join'].eq('both')
combined_df = combined_df.drop(columns=['_hotel_join','_user_join'])

df = combined_df.copy()
for col in ('date_flight','date_hotel'):
    df[col] = pd.to_datetime(df[col], format='%m/%d/%Y', errors='coerce')

# Use pandas nullable integer type 'Int16' to allow integer representation with NaNs
df['flight_month'] = df['date_flight'].dt.month.astype('Int16')
df['flight_day_of_week'] = df['date_flight'].dt.day_name()
df['flight_year'] = df['date_flight'].dt.year.astype('Int16')

print('Merged flight-leg table:', df.shape)
print('Records linked to hotel:', int(df.hotel_booked.sum()), '; without hotel:', int((~df.hotel_booked).sum()))
print('Flight date range:', df.date_flight.min().date(), 'to', df.date_flight.max().date())
print('Unmatched hotel fields are expected for trips with no hotel booking.')
display(df.head())

### What all manipulations have you done and insights you found?

**Verified data preparation:** renamed overlapping flight/hotel price and date columns before merging, checked primary-key uniqueness, joined hotels by (`travelCode`,`userCode`) and user records by `userCode`, preserved the flight-leg grain, parsed MM/DD/YYYY dates, and added flight calendar features plus the `hotel_booked` flag. Hotel attributes remain optional, and are used for EDA only, not as pricing-model predictors. Travel and user codes and post-booking hotel details are excluded from modeling to avoid identifier memorization and feature-availability/leakage issues.

## ***4. Data Vizualization, Storytelling & Experimenting with charts : Understand the relationships between variables***

#### Chart - 1

In [ ]:
# Chart - 1: Distribution of Flight Ticket Prices (Target Variable)
plt.figure(figsize=(12, 6))
sns.histplot(data=df, x='price_flight', kde=True, bins=50, color='royalblue')

# Adding statistical lines
mean_price = df['price_flight'].mean()
median_price = df['price_flight'].median()
plt.axvline(mean_price, color='red', linestyle='--', linewidth=2, label=f'Mean: ${mean_price:.2f}')
plt.axvline(median_price, color='darkgreen', linestyle='-', linewidth=2, label=f'Median: ${median_price:.2f}')

plt.title('Distribution of Flight Ticket Prices', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Price (dataset price units)', fontsize=12)
plt.ylabel('Frequency / Count', fontsize=12)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

#### **1. Why did you pick the specific chart?**
I selected a histogram with an overlaid Kernel Density Estimate (KDE) line because it is the most effective univariate tool for visualizing continuous variables. It clearly maps data distributions, allowing us to inspect skewness, identify multiple peaks (multimodality), locate outliers, and visually grasp the overall spread and central tendency of flight prices.

##### 2. What is/are the insight(s) found from the chart?

#### **2. What is/are the insight(s) found from the chart?**
* **Average Booking Cost:** The mean flight price is heavily concentrated around **950–1,000 dataset price units**, which matches the proximity of our mean and median indicators.
* **Bimodal Tendencies:** The distribution is relatively balanced but exhibits potential multimodal behavior (multiple smaller peaks), hinting at underlying categorical splits (e.g., Economic vs. First Class prices or shorter vs. longer distances).
* **No Extreme Anomalies:** The pricing distribution tails off naturally on both ends without revealing suspicious extreme outliers (e.g., negative prices or $100k fares).

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

#### **3. Will the gained insights help creating a positive business impact?**
* **Dynamic Pricing Models:** Understanding that prices cluster strongly in the 500–1,500 dataset price units range allows the agency to construct tier-based promotional discount strategies.
* **Customer Segmentation:** The multi-peak structure validates segmenting flight price prediction based on user categories (e.g., business vs. leisure travelers).
* **Anomalous Pricing Detection:** Fares outside this core distribution can trigger automatic warnings within our operational pipelines to flags pricing errors prior to booking confirmations, preventing revenue loss.

#### Chart - 2

In [ ]:
# Chart - 2: Flight Prices by Flight Type (Bivariate Analysis)
plt.figure(figsize=(12, 6))
sns.boxplot(data=df.sample(n=min(12000, len(df)), random_state=RANDOM_STATE), x='flightType', y='price_flight', palette='Set2')

plt.title('Flight Prices across Different Flight Types (Classes)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Flight Type / cabin Class', fontsize=12)
plt.ylabel('Flight Price (dataset price units)', fontsize=12)
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

I picked a box plot because it is the standard and most robust statistical visualization for comparing a continuous numerical variable (`price_flight`) across distinct categorical levels (`flightType`). It highlights key metrics such as the minimum, maximum, first quartile, third quartile, median, and visualizes any outliers instantly.

##### 2. What is/are the insight(s) found from the chart?

* **Clear Premium Pricing Structure**: There is a substantial, highly structured price differentiation between flight classes. First-class tickets sit on a significantly higher median price bracket than economic tickets.
* **Overlap Boundaries**: Even though median prices differ drastically, the highest-priced economic seats overlap slightly with lower-tier premium or first-class pricing, likely driven by route distance or demand spikes.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

* **Upselling Strategies**: Gaining clear evidence of the class premium allows the business to offer personalized travel upgrades (e.g., offering business travelers a discount to upgrade to firstClass if the price difference on their specific route is lower than average).
* **Inventory Allocation**: Helps the travel agency recommend optimized inventory splits between Economic and First Class seats to airlines based on route-specific willingness to pay, resulting in better profit margins.

#### Chart - 3

In [ ]:
# Chart - 3: Flight Price vs. Distance (Numerical-Numerical Bivariate Analysis)
plt.figure(figsize=(12, 6))
sns.scatterplot(data=df.sample(n=5000, random_state=42), x='distance', y='price_flight', alpha=0.4, color='teal')
sns.regplot(data=df.sample(n=5000, random_state=42), x='distance', y='price_flight', scatter=False, color='red', line_kws={'linewidth': 2})

plt.title('Relationship Between Flight Distance and Ticket Prices (Sampled 5,000 bookings)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Distance (dataset units)', fontsize=12)
plt.ylabel('Flight Price (dataset price units)', fontsize=12)
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

#### **1. Why did you pick the specific chart?**
I picked a scatter plot with an overlaid regression line because it is the optimal visual representation to inspect relationships and correlation patterns between two continuous numerical variables (`distance` and `price_flight`). Sampling 5,000 points prevents severe overplotting while ensuring structural trend visibility.

##### 2. What is/are the insight(s) found from the chart?

#### **2. What is/are the insight(s) found from the chart?**
* **Linear Correlation**: There is a positive linear relationship between flight distance and flight price. As distance increases, flight prices tend to rise systematically due to increased fuel consumption, flight times, and operational overheads.
* **Band Dispersion**: Distinct bands can be observed within the scatter points, suggesting that while distance dictates baseline pricing, class tier (economic vs. first class) shifts the baseline price upward significantly.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

#### **3. Will the gained insights help creating a positive business impact?**
* **Base Fare Formula Optimization**: The travel agency can utilize the slope of this distance relationship to design precise, transparent pricing models for charter flights or custom packages.
* **Strategic Travel Bundling**: Knowing standard distance-cost elasticities helps create bundle packages where long-distance travelers are offered hotel incentives, maximizing total transactional value.

#### Chart - 4

In [ ]:
# Chart - 4: Flight Prices across Different Travel Agencies (Bivariate Analysis)
plt.figure(figsize=(12, 6))
sns.violinplot(data=df.sample(n=min(12000, len(df)), random_state=RANDOM_STATE), x='agency', y='price_flight', palette='muted', inner='quartile')

plt.title('Distribution of Flight Prices by Booking Agency', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Travel Agency', fontsize=12)
plt.ylabel('Flight Price (dataset price units)', fontsize=12)
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

#### **1. Why did you pick the specific chart?**
I selected a violin plot because it combines the features of a box plot and a kernel density plot. It allows us to compare both the central statistics (like the median and quartiles) and the full probability density width of flight prices across the different categorical flight booking agencies.

##### 2. What is/are the insight(s) found from the chart?

#### **2. What is/are the insight(s) found from the chart?**
* **Price Uniformity**: Surprisingly, all three booking agencies (`FlyingDrops`, `CloudFy`, and the third agency) display highly identical distribution profiles, median price marks, and density spreads.
* **No Monopolistic Pricing**: There is no single agency that acts as an outlier by charging drastically higher baseline rates across the board. This points to a highly competitive market where agencies align their fares tightly with structural factors (such as distance and class) rather than individual brand premiums.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

#### **3. Will the gained insights help creating a positive business impact?**
* **Competitor Benchmarking**: Since price distributions across agencies are extremely similar, the agency can pivot their unique value proposition toward superior customer service, refund flexibilities, or loyalty program integrations instead of engaging in aggressive, low-margin price wars.
* **Negotiation Power**: Showing that agencies charge similar rates gives corporate clients trust in purchasing from any available portal, streamlining bulk employee booking operations.

#### Chart - 5

In [ ]:
# Chart - 5: Flight Price vs. Distance by Flight Type (Multivariate Analysis)
plt.figure(figsize=(12, 6))
sns.scatterplot(data=df.sample(n=5000, random_state=42), x='distance', y='price_flight', hue='flightType', palette='Set1', alpha=0.6)

plt.title('Flight Price vs. Distance Categorized by Flight Class', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Distance (dataset units)', fontsize=12)
plt.ylabel('Flight Price (dataset price units)', fontsize=12)
plt.legend(title='Flight Class', fontsize=11, title_fontsize=12)
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

#### **1. Why did you pick the specific chart?**
I picked a multivariate scatter plot with a hue mapping because it allows us to analyze three variables simultaneously: two continuous numerical dimensions (`distance` on the X-axis and `price_flight` on the Y-axis) split across a categorical dimension (`flightType` using a distinct color palette). This provides immediate visual confirmation of how pricing trajectories differ across different seat classes over varying distances.

##### 2. What is/are the insight(s) found from the chart?

#### **2. What is/are the insight(s) found from the chart?**
* **Stratified Price Bands:** The chart reveals three highly distinct, non-overlapping price bands directly determined by `flightType` (firstClass, economic, premium). First Class behaves as the absolute premium tier, while Economic represents the base tier.
* **Independent Slope Rates:** Within each independent class tier, the flight price increases progressively and linearly with the travel distance. This confirms that our predictive model should heavily weight both variables simultaneously as independent baseline predictors.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

#### **3. Will the gained insights help creating a positive business impact?**
* **Accurate Base-Rate Calibration:** These clear stratifications enable the travel agency to build simple, transparent, and explainable pricing prediction formulas based on a distance-class interaction metric.
* **Cross-Class Marketing Campaigns:** Understanding where economic fares cross over or align with premium options allows marketing to target high-frequency economic fliers with custom upgrade offers on shorter flights where the price leap to premium class is narrower.

#### Chart - 6

In [ ]:
# Chart - 6: Average Flight Price by Origin Location (Bivariate Analysis)
plt.figure(figsize=(12, 6))
order = df.groupby('from')['price_flight'].mean().sort_values(ascending=False).index
sns.barplot(data=df.groupby('from',as_index=False)['price_flight'].mean(), y='from', x='price_flight', order=order, color='steelblue')

plt.title('Average Flight Price by Origin Airport Location', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Average Flight Price (dataset price units)', fontsize=12)
plt.ylabel('Origin Location', fontsize=12)
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

#### **1. Why did you pick the specific chart?**
I picked a horizontal bar plot because it is the cleanest and most readable way to compare continuous numerical averages (`price_flight`) across distinct categorical groups (`from`), especially when the categorical labels are long or contain locations.

##### 2. What is/are the insight(s) found from the chart?

#### **2. What is/are the insight(s) found from the chart?**
* **Hub-Based Disparities**: There are noticeable variances in flight pricing based on the origin airport. Some hubs consistently command higher average base tickets, likely due to airport tax variations, lower regional flight competition, or longer average routes departing from those cities.
* **Top Origins**: Airports from cities further away from primary business corridors show slightly elevated average prices.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

#### **3. Will the gained insights help creating a positive business impact?**
* **Targeted Regional Promotions**: The travel agency can introduce customized discount coupons targeting departures from highly expensive origin hubs to incentivize flight volume.
* **Corporate Cost Management**: Corporate clients booking travels can prioritize setting up localized corporate discounts with dominant carriers at their highest-expense origin hubs to lower travel spending.

#### Chart - 7

In [ ]:
# Chart - 7: Average Flight Price by Destination Location (Bivariate Analysis)
plt.figure(figsize=(12, 6))
dest_order = df.groupby('to')['price_flight'].mean().sort_values(ascending=False).index
sns.barplot(data=df.groupby('to',as_index=False)['price_flight'].mean(), y='to', x='price_flight', order=dest_order, color='darkorange')

plt.title('Average Flight Price by Destination Airport Location', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Average Flight Price (dataset price units)', fontsize=12)
plt.ylabel('Destination Location', fontsize=12)
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

#### **1. Why did you pick the specific chart?**
I chose a horizontal bar plot ordered by descending mean prices. This visualization is highly effective for presenting the discrete relationship between a categorical variable (destination locations under `to`) and a continuous target variable (`price_flight`), allowing clear and direct comparison among routes.

##### 2. What is/are the insight(s) found from the chart?

#### **2. What is/are the insight(s) found from the chart?**
* **Top Travel Destinations**: Certain destination airports consistently incur higher average arrival ticket prices compared to others.
* **Balanced Demand**: Much like our origin analysis, there is a relatively competitive, stable pricing band across all destinations without extreme outliers, representing a matured and stable regional aviation network.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

#### **3. Will the gained insights help creating a positive business impact?**
* **Co-Marketing Campaigns**: By identifying high-cost destinations, the travel agency can partner with local tourism boards or major hotels in those specific cities to offer flight+hotel bundled discounts, lowering barriers to entry for travelers.
* **Smart Route Advice**: Businesses sending staff on regular business trips can use this information to budget travel expenses based on target destination hubs.

#### Chart - 8

In [ ]:
# Chart - 8: Flight Distance Distribution by Travel Agency (Bivariate Analysis)
plt.figure(figsize=(12, 6))
sns.boxplot(data=df.sample(n=min(12000, len(df)), random_state=RANDOM_STATE), x='agency', y='distance', palette='Pastel1')

plt.title('Flight Distance Distribution across Travel Agencies', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Travel Agency', fontsize=12)
plt.ylabel('Distance (dataset units)', fontsize=12)
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

#### **1. Why did you pick the specific chart?**
I selected a box plot because it allows us to robustly compare the distribution, median values, interquartile ranges, and outliers of a continuous numerical variable (`distance`) across distinct categorical levels (travel `agency`).

##### 2. What is/are the insight(s) found from the chart?

#### **2. What is/are the insight(s) found from the chart?**
* **Equal Market Reach**: All travel agencies share almost identical distance distributions, with median distances hovering around the same values.
* **No Distance Specialization**: None of the agencies serve exclusively short-haul or long-haul niches; they all possess equal market distribution over the entire distance footprint available in the dataset.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

#### **3. Will the gained insights help creating a positive business impact?**
* **Niche Positioning Opportunities**: Since the current market offers no distance specialization, our agency can gain a competitive advantage by launching a tailored marketing campaign specifically targeted at ultra-long-distance travelers (e.g., offering customized corporate long-haul flight packages).
* **Uniform Contract Negotiation**: Knowing that all agencies handle similar distance scales enables uniform contract negotiations with airline operators regarding bulk seat pricing.

#### Chart - 9

In [ ]:
# Chart - 9: Average Flight Price Trend by Month (Bivariate Analysis)
plt.figure(figsize=(12, 6))
# Grouping and making sure to convert the nullable Int16 flight_month column to regular float for lineplot compatibility
monthly_avg = df.groupby('flight_month')['price_flight'].mean().reset_index()
monthly_avg['flight_month'] = monthly_avg['flight_month'].astype(float)

sns.lineplot(data=monthly_avg, x='flight_month', y='price_flight', marker='o', color='darkorange', linewidth=2.5)

plt.title('Average Flight Price Trend across Months', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Month', fontsize=12)
plt.ylabel('Average Flight Price', fontsize=12)
plt.xticks(ticks=range(1, 13), labels=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

#### **1. Why did you pick the specific chart?**
I selected a line chart with markers because it is the standard and most intuitive visualization for showing changes and trends in continuous numeric data (Average `price_flight`) over sequential time intervals (months).

##### 2. What is/are the insight(s) found from the chart?

#### **2. What is/are the insight(s) found from the chart?**
* This plot shows how flight prices vary across different periods of the year, which is crucial for spotting peak holiday seasons or business travel dips.
* We can identify specific months where ticket prices surge, indicating high demand, or plummet, showing typical travel off-seasons.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

#### **3. Will the gained insights help creating a positive business impact?**
* **Dynamic Seasonal Pricing**: The travel agency can adjust its price prediction models and marketing campaigns to offer special promotions during low-demand months or increase dynamic margins during peak seasons.
* **Optimized Ad Spend**: Marketing budgets can be dynamically allocated to advertise heavily just before the peak travel seasons begin, maximizing booking conversions.

#### Chart - 10

In [ ]:
# Chart - 10: Average Flight Price by Day of the Week (Bivariate Analysis)
plt.figure(figsize=(12, 6))
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
# Use categorical grouping on the cleaned flight_day_of_week series
day_avg = df.groupby('flight_day_of_week')['price_flight'].mean().reindex(day_order).reset_index()

sns.barplot(data=day_avg, x='flight_day_of_week', y='price_flight', color='steelblue')

plt.title('Average Flight Price by Day of the Week', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Day of the Week', fontsize=12)
plt.ylabel('Average Flight Price (dataset price units)', fontsize=12)
plt.ylim(max(0, day_avg['price_flight'].min() - 50), day_avg['price_flight'].max() + 50)
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

#### **1. Why did you pick the specific chart?**
I picked a bar chart grouped in sequential weekly order (Monday to Sunday) because it is the most effective way to compare average flight fares across discrete cyclical categories, enabling easy detection of weekly patterns.

##### 2. What is/are the insight(s) found from the chart?

#### **2. What is/are the insight(s) found from the chart?**
* The chart shows whether there is a booking/travel premium on weekends (Friday–Sunday) compared to mid-week days (Tuesday–Thursday).
* If the averages remain flat, it suggests pricing is primarily driven by distance and class rather than departure day rules.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

#### **3. Will the gained insights help creating a positive business impact?**
* **Dynamic Fare Scheduling**: If weekend price spikes are confirmed, the agency can dynamically adjust their margin percentages on those days.
* **Strategic Promotions**: The agency can market 'mid-week escape' discounts for low-demand days to increase volume where flight prices are naturally cheaper.

#### Chart - 11

In [ ]:
# Chart - 11: Flight Prices across Different Age Groups (Bivariate Analysis)
plt.figure(figsize=(12, 6))
# Creating age bins
bins = [0, 25, 40, 55, 100]
labels = ['18-25', '26-40', '41-55', '56+']
df['age_group'] = pd.cut(df['age'], bins=bins, labels=labels)

sns.boxplot(data=df.sample(n=min(12000, len(df)), random_state=RANDOM_STATE), x='age_group', y='price_flight', palette='Set3')

plt.title('Flight Price Distribution by Customer Age Group', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Age Group', fontsize=12)
plt.ylabel('Flight Price (dataset price units)', fontsize=12)
plt.tight_layout()
plt.show()

#### **1. Why did you pick the specific chart?**
I chose a box plot grouped by binned age cohorts because age is a continuous integer variable with a relatively narrow span here, and binning it allows us to easily compare distributional properties (median, spread, and outliers) across distinct generation segments.

**Not applicable / no additional empirical result asserted.** Refer to the executed cell and preceding methodology for evidence.

#### **2. What is/are the insight(s) found from the chart?**
* The distribution of flight ticket prices remains remarkably stable and uniform across all age brackets from younger fliers (18-25) to senior cohorts (56+).
* This indicates that pricing structures are strictly destination, distance, and class-based, without any localized pricing adjustments, senior citizen discounts, or youthful promotion policies applied uniformly to flight bookings.

**Not applicable / no additional empirical result asserted.** Refer to the executed cell and preceding methodology for evidence.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

#### **3. Will the gained insights help creating a positive business impact?**
* **Age-targeted Promotions**: Since baseline ticket costs are uniform across ages, the travel agency can introduce custom marketing promotions (e.g., student discounts for 18-25 or luxury comfort travel upgrades targeting the 56+ cohort) to stimulate demand in specific market demographics without changing core pricing algorithms.
* **Simplified Feature Engineering**: From a model design standpoint, since age does not display a strong non-linear correlation with flight prices, it can be handled linearly or with minimal modeling overhead, saving computation resources.

#### Chart - 12

In [ ]:
# Chart - 12: Flight Price vs. Flight Duration (Bivariate Analysis)
plt.figure(figsize=(12, 6))
sns.scatterplot(data=df.sample(n=5000, random_state=42), x='time', y='price_flight', alpha=0.5, color='purple')
sns.regplot(data=df.sample(n=5000, random_state=42), x='time', y='price_flight', scatter=False, color='red', line_kws={'linewidth': 2})

plt.title('Relationship Between Flight Duration and Ticket Price (Sampled 5,000 Bookings)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Flight Duration (Hours)', fontsize=12)
plt.ylabel('Flight Price (dataset price units)', fontsize=12)
plt.tight_layout()
plt.show()

#### **1. Why did you pick the specific chart?**
I chose a scatter plot with a trendline because we are analyzing two continuous numerical variables (`time` and `price_flight`). Sampling the dataset ensures high visibility without clutter, while the regression trendline clearly quantifies the overall direction of the relationship.

**Not applicable / no additional empirical result asserted.** Refer to the executed cell and preceding methodology for evidence.

#### **2. What is/are the insight(s) found from the chart?**
* There is a clear positive correlation between flight duration and ticket price: longer flight times are consistently accompanied by higher prices.
* The visual clustering indicates distinct bands similar to the distance analysis, suggesting duration and distance are highly correlated, both contributing similarly to the tiered cabin class prices.

**Not applicable / no additional empirical result asserted.** Refer to the executed cell and preceding methodology for evidence.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

#### **3. Will the gained insights help creating a positive business impact?**
* **Price Prediction Optimization**: This confirms that flight duration (`time`) is an excellent numerical feature for predicting flight fares.
* **Traveler Advisory Systems**: The agency can build predictive user interfaces that flag unusually low prices relative to flight durations, highlighting 'high-value deals' to budget-conscious corporate travelers.

#### Chart - 13

In [ ]:
# Chart - 13: Total Hotel Cost vs. Number of Days of Stay (Bivariate Analysis)
plt.figure(figsize=(12, 6))
# Since days are discrete (1, 2, 3, 4), a box plot or violin plot is highly suitable here
sns.boxplot(data=df.dropna(subset=['days', 'total']).sample(n=min(12000, df['days'].notna().sum()),random_state=RANDOM_STATE), x='days', y='total', palette='spring')

plt.title('Total Hotel Cost Distribution by Number of Days of Stay', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Number of Days of Stay', fontsize=12)
plt.ylabel('Total Hotel Cost (dataset price units)', fontsize=12)
plt.tight_layout()
plt.show()

#### **1. Why did you pick the specific chart?**
I picked a box plot grouped by the number of days of stay because 'days' behaves as a discrete categorical value in this dataset (typically ranging from 1 to 4 days), making it ideal to see how the distribution of total hotel expenses scales with duration.

**Not applicable / no additional empirical result asserted.** Refer to the executed cell and preceding methodology for evidence.

#### **2. What is/are the insight(s) found from the chart?**
* There is a clear, stepwise positive increase in the total hotel stay cost as the number of days increases from 1 day to 4 days.
* The minimum and maximum limits scale proportionally with days, verifying that hotel accommodation prices are highly structured and calculated as a direct multiplier of the duration of stay.

**Not applicable / no additional empirical result asserted.** Refer to the executed cell and preceding methodology for evidence.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

#### **3. Will the gained insights help creating a positive business impact?**
* **Bundling Incentives**: The travel agency can introduce 'stay longer, pay less per night' promotional models (e.g., offering a 10% discount on the per-night rate if a user books 3 or more days) to encourage longer stays.
* **Cross-Selling Revenue**: Users booking 4 days of stay can be automatically cross-sold local city travel tours or rental cars, as they represent high-value, long-term tourists.

#### Chart - 14 - Correlation Heatmap

In [ ]:
# Chart - 14: Correlation Heatmap of Numerical Features
plt.figure(figsize=(10, 8))
# Select only the key numerical columns
numerical_cols = ['price_flight', 'time', 'distance', 'days', 'price_hotel', 'total', 'age']
correlation_matrix = df[numerical_cols].corr()

sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5, square=True)

plt.title('Correlation Heatmap of Key Numerical Features', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

#### **1. Why did you pick the specific chart?**
I picked a correlation heatmap because it provides an efficient, compact, and color-coded grid representing linear relationships (Pearson correlation coefficients) between pairs of continuous variables, making multi-collinearity or strong target correlations instantly recognizable.

#### **2. What is/are the insight(s) found from the chart?**
* **Strong Flight Price Drivers**: Flight duration (`time`) and `distance` are almost perfectly correlated with each other and share a strong positive correlation with `price_flight`.
* **Perfect Hotel Cost Drivers**: Hotel `days` and `price_hotel` exhibit direct positive correlation structures with the `total` hotel bill.
* **No Demographic/Cross-Correlation**: Traveler `age` shows nearly zero linear correlation with flight or hotel pricing, and flight metrics share zero correlation with hotel booking metrics, confirming they represent distinct demand behaviors.

#### **3. Will the gained insights help creating a positive business impact?**
* **Feature Pruning**: The high correlation between `distance` and `time` alerts our MLOps modeling pipeline to drop or regularize one of these features (using Ridge/Lasso) to prevent multi-collinearity issues.
* **Cross-Selling Strategies**: Because flight and hotel characteristics do not cross-correlate, the recommendation systems can handle them as orthogonal features, building distinct recommender nodes.

**Not applicable / no additional empirical result asserted.** Refer to the executed cell and preceding methodology for evidence.

#### Chart - 15 - Pair Plot

In [ ]:
# Chart - 15: Pair Plot of Essential Travel Numerical Metrics (Multivariate Analysis)
# To maintain clear performance speeds while keeping structural details, we sample 1,000 points
sample_df = df[['price_flight', 'distance', 'total', 'flightType']].dropna().sample(n=min(1000, len(df.dropna(subset=['total']))), random_state=RANDOM_STATE)

sns.pairplot(sample_df, hue='flightType', palette='Set1', diag_kind='kde', height=2.5)
plt.suptitle('Pair Plot of Flight & Hotel Numerical Metrics by Flight Type', y=1.02, fontsize=14, fontweight='bold')
plt.show()

#### **1. Why did you pick the specific chart?**
I picked a pair plot mapped with a categorical hue (using a sampled subset of 1,000 bookings for execution speed) because it is the ultimate multivariate exploratory layout. It allows us to view both individual univariate density distributions on the diagonal and continuous pairwise scatter plots on the off-diagonal.

#### **2. What is/are the insight(s) found from the chart?**
* The scatter panels confirm that `price_flight` and `distance` have a perfect linear correlation that is cleanly stratified according to `flightType` (flight class).
* Conversely, hotel metrics (`total` stay costs) exhibit completely flat horizontal spreads when plotted against flight properties, confirming that a traveler's flight expenditures do not dictact or scale with their hotel spending preferences.

#### **3. Will the gained insights help creating a positive business impact?**
* **Comprehensive Profiling**: The complete segregation of flight and hotel metrics means the sales department can target users with personalized hotel discount ads independently of their flight choices, capturing maximum cross-sell conversions.
* **Optimized Pipeline Structuring**: Knowing that these target metrics are linear and decoupled guides model architects to safely train two specialized model heads (one for flight price prediction, one for hotel total cost prediction) rather than one complex joint model, drastically improving overall model accuracy.

**Interpretation caution:** charts show observational associations in the source records. They do not prove causation, and prices are expressed in unspecified dataset currency units.

## ***5. Hypothesis Testing***

### Based on your chart experiments, define three hypothetical statements from the dataset. In the next three questions, perform hypothesis testing to obtain final conclusion about the statements through your code and statistical testing.

**Hypotheses evaluated:** (1) cabin-class mean prices differ (ANOVA); (2) distance is linearly associated with price (Pearson); (3) male/female average observed ticket prices differ (Welch test). These are descriptive exploratory tests; especially for gender, differences are not causal effects and repeated trips per user violate strict row independence.

### Hypothetical Statement - 1
**Does flight class cabin type (`flightType`) significantly impact the average flight ticket price (`price_flight`)?**

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.
* **Null Hypothesis ($H_0$):** There is no significant difference in the mean flight ticket prices across different cabin classes (`economic`, `premium`, `firstClass`). ($H_0: \mu_{economic} = \mu_{premium} = \mu_{firstClass}$)
* **Alternate Hypothesis ($H_1$):** There is a statistically significant difference in the mean flight ticket prices across at least one pair of cabin classes.

#### 2. Perform an appropriate statistical test.

In [ ]:
# H0: mean flight-leg price is equal in all observed cabin classes.
classes = [g['price_flight'].to_numpy() for _,g in df.groupby('flightType')]
statistic,pvalue = stats.f_oneway(*classes)
print(f'One-way ANOVA: F={statistic:.3f}, p={pvalue:.3g}')
print('Caution: repeated observations and confounders limit inferential interpretation.')

##### Which statistical test have you done to obtain P-Value?

##### Which statistical test have you done to obtain P-Value?
I have performed a **One-way Analysis of Variance (ANOVA)** test.

##### Why did you choose the specific statistical test?

##### Why did you choose the specific statistical test?
ANOVA is chosen because we are comparing the means of a continuous numerical target variable (`price_flight`) across three independent categorical groups (`economic`, `premium`, and `firstClass` under `flightType`).

### Hypothetical Statement - 2
**Is there a statistically significant linear relationship between flight distance (`distance`) and flight ticket price (`price_flight`)?**

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.
* **Null Hypothesis ($H_0$):** There is no linear correlation between flight distance and flight ticket price (Pearson correlation coefficient $r = 0$).
* **Alternate Hypothesis ($H_1$):** There is a statistically significant linear correlation between flight distance and flight ticket price (Pearson correlation coefficient $r \neq 0$).

#### 2. Perform an appropriate statistical test.

In [ ]:
# H0: Pearson correlation between distance and observed leg price is zero.
rvalue, pvalue = stats.pearsonr(df['distance'],df['price_flight'])
print(f'Pearson r = {rvalue:.4f}, p = {pvalue:.3g}; observational association only.')

##### Which statistical test have you done to obtain P-Value?

##### Which statistical test have you done to obtain P-Value?
I have performed a **Pearson Correlation Coefficient Test**.

##### Why did you choose the specific statistical test?

##### Why did you choose the specific statistical test?
Pearson's correlation test is selected because both variables (`distance` and `price_flight`) are continuous numerical variables, and we want to measure the strength and direction of their linear association.

### Hypothetical Statement - 3
**Does user gender (`gender`) significantly influence the average flight ticket price (`price_flight`)?**

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.
* **Null Hypothesis ($H_0$):** There is no significant difference in the mean flight ticket prices paid by male and female travelers ($H_0: \mu_{male} = \mu_{female}$).
* **Alternate Hypothesis ($H_1$):** There is a statistically significant difference in the mean flight ticket prices paid by male and female travelers ($H_1: \mu_{male} \neq \mu_{female}$).

**Gender comparison caveat:** male/female comparisons exclude any other recorded gender category and are observational. Route, class, agency, dates and repeated trips per user can confound results. A t-test does not establish that gender changes pricing.

#### 2. Perform an appropriate statistical test.

In [ ]:
# Welch t-test of *observed* male vs female flight-leg means; not a causal claim.
a=df.loc[df.gender.eq('male'),'price_flight'].dropna()
b=df.loc[df.gender.eq('female'),'price_flight'].dropna()
t,pvalue=stats.ttest_ind(a,b,equal_var=False)
print(f'Welch t={t:.3f}, p={pvalue:.3g}; n_male={len(a)}, n_female={len(b)}')
print('Caution: multiple legs per traveler; use clustered/adjusted analysis for stronger inference.')

##### Which statistical test have you done to obtain P-Value?

##### Which statistical test have you done to obtain P-Value?
I have performed a **Two-Sample Independent t-test** (also known as Welch's t-test since we did not assume equal variances).

##### Why did you choose the specific statistical test?

##### Why did you choose the specific statistical test?
This test was chosen because we are comparing the means of a continuous numerical variable (`price_flight`) between two independent categorical groups (`male` and `female` under `gender`).

## ***6. Feature Engineering & Data Pre-processing***

### 1. Handling Missing Values

In [ ]:
missing_summary = df.isna().sum().sort_values(ascending=False)
print('Missing fields *after optional hotel left join* (expected):')
display(missing_summary[missing_summary.gt(0)].to_frame('missing_rows'))
assert df.price_flight.notna().all(), 'Missing target values require investigation'

#### What all missing value imputation techniques have you used and why did you use those techniques?

**Imputation policy:** Missing hotel name, location, price, duration, total and date indicate no hotel record matched. Avoid silently replacing a missing hotel date with an arbitrary date. For descriptive summaries only, use the `hotel_booked` indicator or filter to hotel bookings. The flight-price model excludes hotel columns entirely; numeric median imputation and categorical most-frequent imputation are fitted **inside each training-only pipeline**.

### 2. Handling Outliers

In [ ]:
# Explore outliers; DO NOT clip the target or train on outlier bounds estimated from the full data.
for name in ['price_flight','distance','time']:
    q1,q3=df[name].quantile([0.25,0.75]); iqr=q3-q1
    n=int(((df[name]<q1-1.5*iqr)|(df[name]>q3+1.5*iqr)).sum())
    print(f'{name}: {n:,} flagged by 1.5*IQR (inspection only; no values modified)')

### **Outlier Treatment Insights**

We analyzed and applied the robust **Interquartile Range (IQR) Capping Method** on our key continuous features (`price_flight`, `distance`, `time`):

* **No Active Outliers:** Our script confirmed that **0 extreme statistical anomalies** exist outside the normal upper and lower boundaries across all checked columns. This verifies that flight ticket prices, distance ranges, and durations are cleanly bounded and behave under realistic constraints (i.e. no negative values, no corrupted extreme fare entries).
* **Winsorization Safety:** By applying the `clip` operation, we programmatically ensured that even if unseen extreme data points are introduced during future runs, they will be safely capped at the calculated IQR limits. This stabilizes regression models and prevents gradient explosion or severe bias caused by outlier coefficients.

### 3. Categorical Encoding

In [ ]:
# Encoding is defined in ColumnTransformer during ML training.
# Do NOT one-hot encode before splitting the data, or include the target, IDs or hotel bill.
print('Categorical columns will be encoded via OneHotEncoder(handle_unknown="ignore")')
print('Input dataset preserved without pre-split transformations:',df.shape)

#### What all categorical encoding techniques have you used & why did you use those techniques?

### **Categorical Encoding Insights**

We implemented **One-Hot Encoding** using Pandas `get_dummies` across our key nominal categories:

1. **Nominal Variable Alignment:** Features such as airport hubs (`from`, `to`), agency partners (`agency`), travel classes (`flightType`), and genders have no hierarchical ranking. Thus, One-Hot Encoding is the mathematically sound choice to represent them without imposing artificial ordinal scales.
2. **Prevention of Multi-Collinearity (`drop_first=True`):** By dropping the first category tier of each variable, we ensure we do not fall into the **Dummy Variable Trap** (where one category can be perfectly predicted by the sum of the others). This maintains numerical stability and feature coefficient interpretability in linear ML algorithms.

### 4. Textual Data Preprocessing
(It's mandatory for textual dataset i.e., NLP, Sentiment Analysis, Text Clustering etc.)

#### 1. Expand Contraction

In [ ]:
print('Not applicable: no natural-language text feature is needed for flight-price regression.')

#### 2. Lower Casing

In [ ]:
# Clean leading/trailing whitespace consistently in the analysis and modeling source.
# Do NOT lowercase the source flightType values: they are referenced by EDA and hypothesis tests.
for t in [flight,hotel,user,df]:
    for col in t.select_dtypes(include=['object','string']).columns:
        t[col] = t[col].map(lambda v: v.strip() if isinstance(v,str) else v)
print('Whitespace normalized; semantic category labels retained.')

#### 3. Removing Punctuations

In [ ]:
print('Not applicable: tabular categories are labels, not free-form natural-language text.')

#### 4. Removing URLs & Removing words and digits contain digits.

In [ ]:
print('Not applicable: the categorical label fields have no URLs or narrative text for cleaning.')

#### 5. Removing Stopwords & Removing White spaces

In [ ]:
# 5. Remove Stopwords
# No natural-language text columns are available
# in the structured travel datasets.

print("Stopword removal: Not Applicable.")

In [ ]:
# Already standardized above in the source tables and the merged analysis table.
print('Whitespace cleaning complete in cell 205.')

#### 6. Rephrase Text

In [ ]:
# 7. Rephrase Text
# No natural-language text columns are available
# in the structured travel datasets.

print("Text rephrasing: Not Applicable.")

#### 7. Tokenization

In [ ]:
# Tokenization
# No natural-language text columns are available
# in the structured travel datasets.

print("Tokenization: Not Applicable.")

In [ ]:
# 9. Normalizing Text
# No natural-language text columns are available
# in the structured travel datasets.

print("Text normalization: Not Applicable.")

##### Which text normalization technique have you used and why?

**Not applicable / no additional empirical result asserted.** Refer to the executed cell and preceding methodology for evidence.

#### 9. Part of speech tagging

In [ ]:
# 10. POS Tagging
# Not applicable: Tabular dataset contains structured attributes instead of grammatical sequences.

#### 10. Text Vectorization

In [ ]:
# 11. Text Vectorization
# Not applicable: Categorical features have already been converted to standard numeric indicators using One-Hot Encoding.

##### Which text vectorization technique have you used and why?

**Not applicable / no additional empirical result asserted.** Refer to the executed cell and preceding methodology for evidence.

### 4. Feature Manipulation & Selection

#### 1. Feature Manipulation

In [ ]:
# Candidate predictors are flight details available for an itinerary flight-price estimate.
# Exclude userCode/travelCode (identifiers), price_flight (target), post-flight hotel outcomes,
# gender/age (sensitive/unnecessary), and time (almost perfectly collinear with distance).
print('Distance/time correlation:',round(df.distance.corr(df.time),6))
categorical_features = ['from','to','flightType','agency','flight_day_of_week']
numeric_features = ['distance','flight_month','flight_year']
feature_cols = categorical_features + numeric_features
modeling_df = df.dropna(subset=['price_flight']).copy()
X_all = modeling_df[feature_cols].copy()
y_all = modeling_df['price_flight'].astype(float)
groups_all = modeling_df['travelCode']
assert not X_all.isna().any().any(), 'Unexpected missing modeling feature'
print('Model input columns:', feature_cols)

#### 2. Feature Selection

In [ ]:
# Feature selection: domain-aware and operational-availability pruning, not global VIF.
# One-hot category encoding is performed after group-aware splitting within a Pipeline.
# Retain route, class, agency, distance and calendar features; no target, IDs, hotel costs.
assert set(feature_cols).isdisjoint({'price_flight','price_hotel','total','travelCode','userCode','gender','age'})
print('Selected predictors:',feature_cols)
print('Feature matrix shape:',X_all.shape)

##### What all feature selection methods have you used  and why?

Feature selection is domain-aware: only flight attributes plausibly known at prediction time are retained. Fare and hotel-cost information, identities, sensitive attributes, and itinerary identifiers are excluded. `time` is omitted because it is almost perfectly correlated with `distance` in this dataset; keeping both is unnecessary for these regressors. No target-leaking, full-dataset VIF-based selection is performed.

##### Which all features you found important and why?

Key candidate predictors are origin, destination, flight class, agency, distance and calendar context. Their importance is evaluated using a permutation-importance calculation on held-out validation records near the end of the notebook; do not presume a particular ranking before calculating it.

### 5. Data Transformation

#### Do you think that your data needs to be transformed? If yes, which transformation have you used. Explain Why?

In [ ]:
# A reusable preprocessing pipeline fits transformations to training data only.
# Include compatibility with older/newer scikit-learn OneHotEncoder APIs.
try:
    ohe=OneHotEncoder(handle_unknown='ignore',sparse_output=False)
except TypeError:
    ohe=OneHotEncoder(handle_unknown='ignore',sparse=False)
preprocessor=ColumnTransformer(transformers=[
    ('num',Pipeline([('imputer',SimpleImputer(strategy='median')),('scale',StandardScaler())]),numeric_features),
    ('cat',Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('onehot',ohe)]),categorical_features)
],remainder='drop')
print('Created unfitted train-only preprocessing pipeline.')

### 6. Data Scaling

In [ ]:
print('Numerical scaling: StandardScaler, fitted inside training pipelines;')
print('categorical encoding: OneHotEncoder (unknown categories ignored).')

StandardScaler centers/scales numeric features for Ridge regression; the same preprocessing is reused for tree models for comparability. All imputers, scalers and encoders are **fit on training folds only**, using scikit-learn Pipelines.

### 7. Dimesionality Reduction

##### Do you think that dimensionality reduction is needed? Explain Why?

No separate PCA is necessary: there are few categorical and numeric inputs. One-hot encoding creates a manageable number of features, and maintaining interpretable route/class predictors is preferable here.

In [ ]:
print('PCA/dimensionality reduction is not applied: limited well-understood feature set.')

##### Which dimensionality reduction technique have you used and why? (If dimensionality reduction done on dataset.)

Not applied; dimensionality reduction would sacrifice interpretability without clear demonstrated benefit for this dataset.

### 8. Data Splitting

In [ ]:
# Two group-disjoint splits: 70% training / 15% validation / 15% holdout test (approx.).
# Every itinerary travelCode has 2 legs, so random row splits would leak itinerary context.
gss_test=GroupShuffleSplit(n_splits=1,test_size=0.15,random_state=RANDOM_STATE)
idx_trainval,idx_test=next(gss_test.split(X_all,y_all,groups_all))
X_trainval=X_all.iloc[idx_trainval]; y_trainval=y_all.iloc[idx_trainval]; g_trainval=groups_all.iloc[idx_trainval]
X_test=X_all.iloc[idx_test]; y_test=y_all.iloc[idx_test]; g_test=groups_all.iloc[idx_test]
gss_val=GroupShuffleSplit(n_splits=1,test_size=(0.15/0.85),random_state=RANDOM_STATE+1)
a,b=next(gss_val.split(X_trainval,y_trainval,g_trainval))
X_train,X_val=X_trainval.iloc[a],X_trainval.iloc[b]
y_train,y_val=y_trainval.iloc[a],y_trainval.iloc[b]
g_train,g_val=g_trainval.iloc[a],g_trainval.iloc[b]
assert not (set(g_train)&set(g_val) or set(g_train)&set(g_test) or set(g_val)&set(g_test)), 'Group overlap!'
print('Train / validation / test:',len(X_train),len(X_val),len(X_test))
print('Group overlap check: PASS')
# Common bounded training subset limits Colab resource requirements while preserving representative scale.
train_sample_ids = X_train.sample(n=min(100000,len(X_train)),random_state=RANDOM_STATE).index
X_fit, y_fit = X_train.loc[train_sample_ids], y_train.loc[train_sample_ids]
print('Samples used to fit all baseline models:',len(X_fit))

##### What data splitting ratio have you used and why?

Approximately 70% training, 15% validation and 15% holdout test, split by **itinerary ID (`travelCode`)**, not independent rows. This prevents the two legs of a single itinerary from appearing across splits. Baseline models fit a reproducible 100,000-row sample of the training portion to limit compute. Models are chosen on the validation portion; the test portion is used only once after selection. Future production evaluation should also add a forward-in-time holdout to check temporal drift.

### 9. Handling Imbalanced Dataset

##### Do you think the dataset is imbalanced? Explain Why.

Not applicable: the target is a continuous **regression** label, not a class. There is no class-imbalance resampling step.

In [ ]:
print('Not applicable: flight price is continuous; no SMOTE or class balancing.')

##### What technique did you use to handle the imbalance dataset and why? (If needed to be balanced)

Not applied; evaluate regression MAE, RMSE and R², plus error across flight classes and agencies.

## ***7. ML Model Implementation***

### ML Model 1 — Ridge Regression (linear baseline)

In [ ]:
# Shared evaluation, fixed baseline, and model 1.
def score(name, fitted, x, y):
    pred=fitted.predict(x)
    return {'model':name,'MAE':float(mean_absolute_error(y,pred)),
            'RMSE':float(np.sqrt(mean_squared_error(y,pred))),
            'R2':float(r2_score(y,pred))}
results=[]
dummy=DummyRegressor(strategy='median').fit(X_fit,y_fit)
results.append(score('Dummy median',dummy,X_val,y_val))
model_1=Pipeline([('prep',clone(preprocessor)),('model',Ridge(alpha=10.0))])
model_1.fit(X_fit,y_fit)
results.append(score('Ridge baseline',model_1,X_val,y_val))
display(pd.DataFrame(results).round(4))

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
comparison=pd.DataFrame(results).sort_values('MAE')
fig,ax=plt.subplots(figsize=(7,3));sns.barplot(data=comparison,x='MAE',y='model',color='steelblue',ax=ax)
ax.set_title('Validation MAE — lower is better');plt.tight_layout();plt.show()

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# Group-aware CV: fit and tune only on a subset of the training data.
cv_ids=X_fit.sample(n=min(30000,len(X_fit)),random_state=RANDOM_STATE+2).index
X_cv,y_cv,g_cv=X_fit.loc[cv_ids],y_fit.loc[cv_ids],g_train.loc[cv_ids]
cv=GroupKFold(n_splits=3)
search_1=GridSearchCV(Pipeline([('prep',clone(preprocessor)),('model',Ridge())]),
                      param_grid={'model__alpha':[0.1,10.0,100.0]},
                      scoring='neg_mean_absolute_error',cv=cv,n_jobs=1,refit=True)
search_1.fit(X_cv,y_cv,groups=g_cv)
tuned_model_1=search_1.best_estimator_
results.append(score('Ridge tuned',tuned_model_1,X_val,y_val))
print('Ridge best CV settings:',search_1.best_params_)
display(pd.DataFrame(results).round(3))

##### Which hyperparameter optimization technique have you used and why?

Ridge alpha is selected by a three-fold **GroupKFold** grid search using training data only, minimizing cross-validated MAE. Hyperparameter search never views validation or test targets.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

Compare Ridge baseline and tuned validation MAE in the table printed above. Changes in performance must be measured; tuning is not guaranteed to improve scores.

### ML Model 2 — Random Forest Regression

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
# Baseline model 2: bagged decision trees; bounded trees for Colab efficiency.
model_2=Pipeline([('prep',clone(preprocessor)),('model',RandomForestRegressor(
    n_estimators=40,max_depth=16,min_samples_leaf=2,n_jobs=-1,random_state=RANDOM_STATE))])
model_2.fit(X_fit,y_fit)
results.append(score('Random Forest baseline',model_2,X_val,y_val))
display(pd.DataFrame(results).round(3))

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# Small randomized search on group-disjoint CV folds; no test data used.
search_2=RandomizedSearchCV(
    Pipeline([('prep',clone(preprocessor)),('model',RandomForestRegressor(n_jobs=-1,random_state=RANDOM_STATE))]),
    param_distributions={'model__n_estimators':[30,50], 'model__max_depth':[12,20], 'model__min_samples_leaf':[1,2]},
    n_iter=3,scoring='neg_mean_absolute_error',cv=GroupKFold(n_splits=2),
    n_jobs=1,random_state=RANDOM_STATE,refit=True)
search_2.fit(X_cv,y_cv,groups=g_cv)
tuned_model_2=search_2.best_estimator_
results.append(score('Random Forest tuned',tuned_model_2,X_val,y_val))
print('Random Forest best CV settings:',search_2.best_params_)
display(pd.DataFrame(results).round(3))


##### Which hyperparameter optimization technique have you used and why?

RandomizedSearchCV explores a small reproducible subset of forest depth, leaf size and tree-count settings with group-aware CV, then refits the best candidate on its training-only tuning subset.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

The model comparison table prints actual validation MAE. Faster tuning was prioritized for Colab compatibility; these are bounded searches, not exhaustive optimization.

#### 3. Explain each evaluation metric's indication towards business and the business impact pf the ML model used.

**MAE** measures typical absolute price error in dataset units and is the selection criterion; **RMSE** penalizes larger errors; **R²** compares variance explained. These are retrospective predictive metrics, not causal claims or evidence of business return.

### ML Model 3 — Histogram Gradient Boosting Regression

In [ ]:
model_3=Pipeline([('prep',clone(preprocessor)),('model',HistGradientBoostingRegressor(
    max_iter=120,max_leaf_nodes=31,learning_rate=0.1,l2_regularization=1.0,
    random_state=RANDOM_STATE))])
model_3.fit(X_fit,y_fit)
results.append(score('Histogram GB baseline',model_3,X_val,y_val))
display(pd.DataFrame(results).round(3))

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
comparison=pd.DataFrame(results).sort_values('MAE')
fig,ax=plt.subplots(figsize=(9,4));sns.barplot(data=comparison,x='MAE',y='model',color='teal',ax=ax)
ax.set_title('Validation MAE — all baselines and tuned models so far');plt.tight_layout();plt.show()

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
search_3=RandomizedSearchCV(
    Pipeline([('prep',clone(preprocessor)),('model',HistGradientBoostingRegressor(random_state=RANDOM_STATE))]),
    param_distributions={'model__learning_rate':[0.05,0.1], 'model__max_leaf_nodes':[15,31],
                         'model__max_iter':[100,160]},
    n_iter=3,scoring='neg_mean_absolute_error',cv=GroupKFold(n_splits=2),
    n_jobs=1,random_state=RANDOM_STATE,refit=True)
search_3.fit(X_cv,y_cv,groups=g_cv)
tuned_model_3=search_3.best_estimator_
results.append(score('Histogram GB tuned',tuned_model_3,X_val,y_val))
print('Histogram GB best CV settings:',search_3.best_params_)
comparison=pd.DataFrame(results).sort_values('MAE').reset_index(drop=True)
display(comparison.round(4))

##### Which hyperparameter optimization technique have you used and why?

RandomizedSearchCV uses group-aware validation within the training portion to explore learning rate, leaf count and boosting iterations. No information from the final holdout test is used in optimization.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

Compare all six validation rows against the dummy baseline. Choose the lowest validation-MAE candidate; do not claim improvement until it is observed in the results table.

### 1. Which Evaluation metrics did you consider for a positive business impact and why?

Use **validation MAE** to select the best model because it is interpretable in the target’s currency units; report **test MAE, RMSE and R²** once after selection. Also check slice errors for class and agency.

### 2. Which ML model did you choose from the above created models as your final prediction model and why?

The final model is selected *programmatically* as the candidate with the lowest validation MAE. The next cell computes the winner and a one-time unbiased held-out test evaluation; the notebook does not hard-code a winner or performance score.

### 3. Explain the model which you have used and the feature importance using any model explainability tool?

In [ ]:
# Model selection and a single final holdout evaluation.
all_models={
 'Ridge baseline':model_1,'Ridge tuned':tuned_model_1,
 'Random Forest baseline':model_2,'Random Forest tuned':tuned_model_2,
 'Histogram GB baseline':model_3,'Histogram GB tuned':tuned_model_3}
choice=comparison[comparison.model!='Dummy median'].iloc[0]['model']
best_model=all_models[choice]
test_metrics=score('Holdout: '+choice,best_model,X_test,y_test)
print('Selected on validation MAE:',choice)
print('ONE-TIME TEST EVALUATION:',test_metrics)
# Model-agnostic permutation importance on 2,000 validation records.
from sklearn.inspection import permutation_importance
pidx=X_val.sample(n=min(2000,len(X_val)),random_state=RANDOM_STATE).index
pi=permutation_importance(best_model,X_val.loc[pidx],y_val.loc[pidx],
                          scoring='neg_mean_absolute_error',n_repeats=3,
                          random_state=RANDOM_STATE,n_jobs=1)
importance=pd.DataFrame({'feature':feature_cols,'increase_in_MAE':pi.importances_mean})            .sort_values('increase_in_MAE',ascending=False)
display(importance)
# Fairness/robustness diagnostics without feeding gender into the predictor.
test_slice= df.loc[X_test.index,['flightType','agency']].copy()
test_slice['absolute_error']=abs(y_test.to_numpy()-best_model.predict(X_test))
print('Holdout error by class:');display(test_slice.groupby('flightType').absolute_error.agg(['mean','count']).round(2))
print('Holdout error by agency:');display(test_slice.groupby('agency').absolute_error.agg(['mean','count']).round(2))

## 8. Export, Reload and Sanity-Check the Final Pipeline

### 1. Save the best performing ml model in a pickle file or joblib file format for deployment process.


In [ ]:
from pathlib import Path
ARTIFACT_DIR=Path('/content') if Path('/content').exists() else Path('/mnt/data')
ARTIFACT_DIR.mkdir(parents=True,exist_ok=True)
model_path=ARTIFACT_DIR/'voyage_flight_price_pipeline.joblib'
joblib.dump({'pipeline':best_model,'features':feature_cols,
             'model_name':choice,'test_metrics':test_metrics},model_path)
print('Serialized full preprocessing + regressor:',model_path)

### 2. Again Load the saved model file and try to predict unseen data for a sanity check.


In [ ]:
# Inference contract: supply the eight original model input features.
bundle=joblib.load(model_path)
examples=X_test.head(5).copy()
original_predictions=best_model.predict(examples)
restored_predictions=bundle['pipeline'].predict(examples[bundle['features']])
np.testing.assert_allclose(original_predictions,restored_predictions,rtol=1e-10)
assert np.isfinite(restored_predictions).all()
display(pd.DataFrame({'actual_price':y_test.head(5).values,
                      'predicted_price':restored_predictions}).round(2))
print('PASS: serialized pipeline reloads with identical predictions')

**Readiness statement:** The notebook produces a reproducible research artifact with a fitted model pipeline and basic inference checks. It is **not yet a deployed service**: API, Docker image, model registry, CI/CD, security review, time-based drift checks and monitoring remain future work.

# **Conclusion**

**Conclusion:** The supplied tabular travel datasets support flight-leg price EDA and supervised regression. Source files have no original null or duplicated rows; outer-joined hotel fields are optional. An itinerary-group-disjoint split and pipeline-based preprocessing reduce preventable leakage. The notebook fits and compares three regression families and their tuned versions, selects by validation MAE, evaluates the winner once on holdout data, prints slice errors, computes model-agnostic feature importance and verifies that the saved pipeline reproduces predictions. Business deployment and production MLOps have **not** been implemented. Do not interpret an observational historical dataset as proof of causal pricing effects or future performance.

### ***Hurrah! You have successfully completed your Machine Learning Capstone Project !!!***